# Week 11: NLP

In this workshop you build a **RAG system**, retrieval-augmented generation, over every NUS module:

- **Section 1 - Intro to NLP and LLMs:** why text is hard for computers, how an LLM writes, and where it gets things wrong.
- **Section 2 - Retrieval Engine:** embeddings, chunking and semantic search over the NUSMods data.
- **Section 3 - Generation + Evaluation:** let Gemini answer from the retrieved modules, then measure how good the retrieval is.

Run the cells from top to bottom. Cells marked **YOUR TURN** are for you to edit.

# Section 1 - Intro to NLP and LLMs

By the end of this workshop you will have built a **RAG system**: a search engine over every NUS module, connected to an LLM that answers questions using what the search finds. This section follows the slides:

1. **Reading text:** preprocessing, tokens and the token budget.
2. **Feature extraction:** keyword search, Bag of Words, TF-IDF and BM25.
3. **Hallucination and RAG:** ask Gemini about NUS modules, check its answers against the real data, and see why RAG is the fix.

## Setup

Run the cells in this part now, while everyone is arriving.

**In VS Code:** click **Select Kernel** (top right of the notebook) → **Python Environments**, and pick the virtual environment where you ran `pip install -r requirements.txt` (run it inside the `week-11-nlp` folder, where `requirements.txt` is). If VS Code asks to install `ipykernel`, click **Install**.

The first cell loads three things through `rag_utils.py`, a small helper file that all three sections share:

- `modules`: every module NUS offers in AY 2026-2027, taken from [NUSMods](https://nusmods.com). Each module is a Python dictionary with its code, title, description, workload and more. The organisers downloaded this in advance and saved it in `data/`, so the workshop does not depend on NUSMods being up.
- `embeddings`: one row of numbers per module. You will meet these in Section 2.
- `model`: `all-MiniLM-L6-v2`, a small model that turns text into numbers. It runs on your own CPU and needs no API key. The first run downloads it (about 90 MB).

In [ ]:
# %pip install -q google-genai sentence-transformers numpy matplotlib
import json
import os
import re
import textwrap
import urllib.request
import warnings

os.environ["TRANSFORMERS_VERBOSITY"] = os.environ["HF_HUB_VERBOSITY"] = "error"  # hide harmless warnings
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

import rag_utils
from rag_utils import build_chunk, keyword_search, search

modules, embeddings, model = rag_utils.load()
by_code = {module["moduleCode"]: module for module in modules}  # "CS4248" -> its record

print(f"{len(modules)} modules offered in AY {rag_utils.AY}, fetched from NUSMods on {rag_utils.FETCH_DATE}")

In [ ]:
import pandas as pd

# One row per module. workload = weekly hours of [lecture, tutorial, lab, project, preparation]
pd.DataFrame(modules).head(5)

### Connect Gemini

Each of you needs your **own** free API key: the free quota is counted per Google Cloud project, so 40 people sharing one key would run out within minutes.

> **Your Gemini API key (one-time, about 2 minutes)**
> 1. Go to **[aistudio.google.com/apikey](https://aistudio.google.com/apikey)**, sign in with a *personal* Google account, click **Create API key** and copy the key. It's free — **don't** set up billing, because that turns the free tier off for the project.
> 2. Paste the key between the quotes in the cell below, then run it.
>
> The key is saved inside the notebook file, so don't share or commit the notebook with your key in it.

The cell connects `ask_gemini` (from `rag_utils.py`), the one function we use to talk to Gemini. It sends your prompt and nothing else: no extra documents, and no Google Search. If the call fails, for example because the free quota has run out, it prints the reason and returns a saved answer instead of crashing.

You should see `Gemini says: ready`. If you see `Gemini call failed` instead, the message tells you why. The most common one is `API key not valid`: copy the key again and re-run the cell. If it still fails, raise your hand. You can follow the Gemini demos on the main screen in the meantime.

In [ ]:
from rag_utils import ask_gemini

GEMINI_API_KEY = ""  # vscode will prompt you to enter this in a secure way, or you can hardcode it here if you prefer

rag_utils.connect_gemini(GEMINI_API_KEY)
print("Gemini says:", ask_gemini("Reply with exactly: ready"))

## 1.1 Reading text

How does a computer read text? Computers only understand numbers, so we have to turn words into numbers. That is hard, because text is **messy** (punctuation, capital letters, codes like `CS3244`) and **ambiguous** (one word can mean different things, and different words can mean the same thing).

NLP deals with this through a pipeline. This section covers the first six steps, and Section 2 the last two:

**Raw text → Cleaning → Tokenization → Normalization → Chunking → Feature extraction** → Embedding → Retrieval

Our example all the way through is one real NUSMods description, the first sentence of **CS3244 Machine Learning**:

> This course introduces basic concepts and algorithms in machine learning and neural networks.

## 1.2 Preprocessing

Before a computer can count words, we clean the text up in three stages:

- **Cleaning:** remove punctuation and symbols.
- **Tokenization:** split into words.
- **Normalization:** make different forms of the same word look alike. That covers lowercasing, stopword removal (drop common words such as "this", "and", "in"), stemming (chop off word endings: `machine` → `machin`) and lemmatization (back to the dictionary form: `networks` → `network`).

These steps help methods that count words, such as BM25 in 1.4. MiniLM, the model we use in Section 2, does not need most of them: it lowercases by itself and has its own tokenizer.

## 1.3 Tokenization

A **token** is a fundamental chunk of text that a language model processes, and **tokenization** is splitting text into tokens. Common words are usually 1 token, while rare words and codes get split into smaller pieces, e.g. `CS3244` → `CS | 324 | 4`.

A model can only read a limited number of tokens at once: its **token budget**. Anything past the budget gets cut off, and the model never sees it. **Chunking** cuts long text into pieces that fit the budget. You will choose chunks yourself in Section 2.

Here is MiniLM's tokenizer reading the CS3244 sentence. Change `budget` and re-run to see where the sentence gets cut.

In [ ]:
sentence = by_code["CS3244"]["description"].split(". ")[0] + "."
tokens = model.tokenizer.tokenize(sentence)
print(f"{len(tokens)} tokens:", tokens)

# Try it: change the budget, then re-run
budget = 10

print()
print(f"Read ({budget} tokens):", model.tokenizer.convert_tokens_to_string(tokens[:budget]))
print(f"Cut off ({len(tokens[budget:])} tokens):", model.tokenizer.convert_tokens_to_string(tokens[budget:]))
print(f"MiniLM's real budget: {model.max_seq_length} tokens")

With a budget of 10, the model never finds out that CS3244 covers neural networks. Real budgets are bigger, but so is real text.

**YOUR TURN:** every model comes with its own tokenizer. Alter the words below and re-run to see how MiniLM splits them, then paste the same words into ChatGPT's tokenizer at [tiktokenizer.vercel.app](https://tiktokenizer.vercel.app/) and compare. Try other module codes, your name, Singlish, or another language.

**Think about it:** how does the model decide where to split?

In [ ]:
# YOUR TURN: alter the words, then re-run
my_words = [
    "learning", "GPU", "CS3244", "NUSMods",      # from the slides
    "regularities", "self-learning", "CS2040S",  # from the CS3244 description, and one of its prerequisites
    "eﬃciency", "汉语拼音方案",                    # oddities from the FIN3702A and CL3211 descriptions
]

for word in my_words:
    print(f"{word:>15} -> {model.tokenizer.tokenize(word)}")

In [ ]:
# The model only sees ID numbers, and decode turns the numbers back into text
# Try it: change the text, then re-run
text = "CS3244 introduces neural networks"

ids = model.tokenizer.encode(text, add_special_tokens=False)
print("Token IDs:", ids)
print("Tokens:   ", model.tokenizer.convert_ids_to_tokens(ids))
print("Decoded:  ", model.tokenizer.decode(ids))

### Subword encoding

Storing every whole word would take too many pieces, and still fail on a new word such as "NUSMods". Single letters can spell anything, but make text far too long. **Subwords** sit in between: break words into smaller pieces that can be reused.

- Common words stay whole: `course` → `[course]`.
- Rare words are built from known pieces: `CS3244` → `[CS | 324 | 4]`.

So the model can read new words with a fixed list of pieces. Two common ways to build that list:

- **Byte-Pair Encoding (BPE):** start from single letters, and keep gluing together the two pieces that appear side by side most often. On `learning`, `training` and `mining`, round 1 glues `i + n → in`, and round 2 glues `in + g → ing`. New words reuse the pieces: `coding` → `c | o | d | ing`. ChatGPT's tokenizer is a BPE tokenizer.
- **WordPiece:** BPE's cousin. Same idea, but BPE glues the most common pair, while WordPiece glues the pair that sticks together most (pieces that rarely show up apart). Pieces inside a word start with `##`: `GPU` → `[gp | ##u]`.

MiniLM uses WordPiece, which is where the `##` in your output comes from. It also lowercases everything first, which is why decoding gives back `cs3244`, not `CS3244`. `[UNK]` means **unknown**: the character is not in MiniLM's vocabulary at all. `eﬃciency` looks normal, but its "ﬃ" is one special character (probably copied from a PDF), so MiniLM cannot read the word. That is messy text in action.

## 1.4 Feature extraction

### The search process

RAG has to find the right modules first. The simplest way is **keyword search**: return every module whose title contains your exact words, like the search bar on NUSMods. It is exact, but it ignores similar modules we might want: "machine learning" finds CS3244 Machine Learning, but not AIS5103 Foundations of Deep Learning.

The idea: look at all the words in the description, not just the title. `keyword_search` (from `rag_utils.py`) does that, like Ctrl+F on every description.

**YOUR TURN:** change the query and compare the two searches.

In [ ]:
def title_search(query):
    """Modules whose title contains the query, word for word."""
    return [module for module in modules if query.lower() in module["title"].lower()]


# YOUR TURN: change the query, then re-run
query = "machine learning"

for where, matches in [("titles", title_search(query)), ("descriptions", keyword_search(query))]:
    print(f'"{query}" in {where}: {len(matches)} modules')
    for match in matches[:5]:
        print(f"    {match['moduleCode']:<9} {match['title']}")

The descriptions give many more matches, AIS5103 among them, but two problems remain. The matches come back in alphabetical order, not by relevance. And both searches compare characters: try `"how computers understand language"`, and neither finds anything, because no title or description contains that exact phrase.

### Bag of Words, TF-IDF and BM25

To rank modules, we turn each description into numbers (**feature extraction**) and give every module a score:

- **Bag of Words (BoW):** count each word in a description, so the text becomes a row of numbers. The score is the count of shared words. Problems: every word counts the same, so "course", which is in almost every description, still earns points; and word order is lost ("students teach tutors" = "tutors teach students").
- **TF-IDF:** TF (term frequency) is how often the word appears in this description, divided by its length. IDF (inverse document frequency) is how rare the word is across all descriptions: $\log_{10}(N \div \text{descriptions that have the word})$. TF × IDF is high for a word that is frequent here and rare elsewhere.
- **BM25 (Best Matching 25):** TF-IDF with two fixes. **Saturation:** the 10th "learning" adds much less than the 1st (`k1 = 1.2`). **Length:** long descriptions don't win just by having more words (`b = 0.75`).

The next cell scores every NUSMods description all three ways, with the same formulas as the tables on the slides.

In [ ]:
from collections import Counter


def split_words(text):
    """Lowercase the text and split it into words."""
    return re.findall(r"[a-z0-9]+", text.lower())


bags = [Counter(split_words(module["description"])) for module in modules]  # one Bag of Words per module
lengths = np.array([sum(bag.values()) for bag in bags])                       # words in each description
has_word = Counter(word for bag in bags for word in bag)                      # descriptions that have each word


def word_scores(query, method="bm25", k1=1.2, b=0.75):
    """Score every description against the query with "bow", "tfidf" or "bm25"."""
    scores = np.zeros(len(modules))
    for word in split_words(query):
        count = np.array([bag[word] for bag in bags])
        idf = np.log10(len(modules) / has_word[word]) if has_word[word] else 0.0
        if method == "bow":
            scores += count
        elif method == "tfidf":
            scores += count / lengths * idf
        elif method == "bm25":
            scores += idf * count * (k1 + 1) / (count + k1 * (1 - b + b * lengths / lengths.mean()))
    return scores


def word_search(query, method="bm25", k=5):
    """Return the top k (module, score) pairs."""
    scores = word_scores(query, method)
    top = np.argsort(-scores)[:k]
    return [(modules[i], float(scores[i])) for i in top]

In [ ]:
# YOUR TURN: change the query, then re-run
query = "machine learning course"

for method in ["bow", "tfidf", "bm25"]:
    print(method)
    for module, score in word_search(query, method):
        print(f"  {score:6.2f}  {module['moduleCode']:<9} {module['title']:<45} ({len(split_words(module['description']))} words)")

**How to read it:** look at the word counts in brackets. A typical description has about 90 words.

- **BoW** puts EBA5012 first, a 243-word description: long text repeats the query words more often.
- **TF-IDF** divides by the length, so short descriptions such as SA4110 (17 words) jump to the top.
- **BM25** caps what repeated words add and compares each length with the average, so long and short descriptions compete on fairer terms.

"course" adds almost nothing to TF-IDF and BM25: nearly every description has it, so its IDF is close to 0. Try your own query, such as `"how computers understand language"`. Do the three methods agree?

## 1.5 Hallucination

When we use an LLM to look something up, it might **not know the updated data**, or **invent fluent but untrue facts**. Two reasons:

- **Knowledge cutoff:** it only knows what it was trained on, up to a certain date.
- **Next-token prediction:** it writes one token at a time, picking what sounds likely, not what is true.

Confident answers that are wrong are called **hallucinations**. We will ask Gemini about NUS modules, giving it **no extra information**, and then **check its answer** against the real NUSMods data.

`verify` does the checking. It finds every module code in an answer, such as `CS3244`, and looks it up in this year's module list.

In [ ]:
MODULE_CODE = re.compile(r"\b[A-Z]{2,4}\d{4}[A-Z]{0,2}\b")  # e.g. CS3244, CS2040S


def verify(answer):
    """Look up every module code in the answer in the real NUSMods data."""
    codes = list(dict.fromkeys(MODULE_CODE.findall(answer)))  # unique, in order
    for code in codes:
        real = by_code.get(code)
        if real:
            print(f"  ✅ {code:<9} real title: {real['title']}")
        else:
            print(f"  ❌ {code:<9} not offered in AY {rag_utils.AY}, or does not exist")
        said = next(line for line in answer.splitlines() if code in line)  # where Gemini mentioned it
        print(f"{'':15}Gemini wrote: {textwrap.shorten(re.sub(r'[*_#`|>]', '', said), 90)}")
    print()
    print(f"{sum(code in by_code for code in codes)} of {len(codes)} module codes are real.")

In [ ]:
question = "Which NUS modules cover attention mechanisms? List the module codes and titles."

answer = ask_gemini(question)
display(Markdown(answer))
print("-" * 60)
verify(answer)

**How to read it:**

- ❌ means the code is not in this year's NUSMods data. Either the module does not exist, or it is not running this year.
- ✅ with a different title means the code is real, but Gemini attached the wrong module to it, or used an old name.
- ✅ with a matching title means Gemini got that one right. You only know because you checked.

### Without and with retrieval

The surest way to catch an LLM inventing things is to ask about something that does not exist. `CS3246` looks like a real CS module, right next to CS3244, but NUS has never offered it. The cell below first checks NUSMods, then asks Gemini twice: once on its own, and once with what NUSMods has pasted into the prompt.

In [ ]:
module_code = "CS3246"  # made up: NUS has never offered a module with this code
module_question = (f"What is the NUS module {module_code} about? "
                   "Give its title, a short description and its weekly workload in hours.")

print(f"Is {module_code} in NUSMods AY {rag_utils.AY}?", module_code in by_code)
print()
print("WITHOUT RETRIEVAL:")
print(ask_gemini(module_question))
print()

# What NUSMods has: the module's entry, or a note that there is none
if module_code in by_code:
    entry = build_chunk(by_code[module_code])
else:
    entry = f"NUSMods has no module {module_code} in AY {rag_utils.AY}."
print("WITH WHAT NUSMODS HAS:")
print(ask_gemini(f"Answer using only this information from NUSMods.\n\n{entry}\n\n{module_question}"))

Without retrieval, Gemini may describe a module that does not exist, with a title, topics and workload hours that all sound reasonable: that is exactly what next-token prediction produces. With what NUSMods has in front of it, Gemini can tell you the module does not exist, and you know where that answer came from: NUSMods. Hold on to that idea, because it is the core of 1.6. Change `module_code` to a module you have taken, such as `CS2040S`, to see both answers for a real module.

**YOUR TURN:** ask Gemini your own question about NUS modules, then check it. Ask for module codes, so that `verify` has something to check. Some things to try:

- A topic from the CS3244 description: *"Which NUS modules teach neural networks?"*
- What comes next: *"Which NUS modules should I take after CS3244?"*
- Something that changes every year, to test the knowledge cutoff: *"Which modules are precluded for CS3244?"*

In [ ]:
# YOUR TURN: ask your own question, then re-run
my_question = "Which NUS modules should I take after CS3244 Machine Learning? List the module codes and titles."

my_answer = ask_gemini(my_question)
display(Markdown(my_answer))
print("-" * 60)
verify(my_answer)

## 1.6 The concept of RAG

**Retrieval-Augmented Generation (RAG)** is a framework that improves an LLM by connecting it to an external knowledge source. Ours is NUSMods.

Why not paste all of NUSMods into the prompt, the way we pasted the CS3244 entry? Because of the **token budget**: there are too many descriptions to fit into a single prompt. The cell below counts tokens with MiniLM's tokenizer, so treat the numbers as estimates.

In [ ]:
chunks = [build_chunk(m) for m in modules]
chunk_lengths = [len(ids) for ids in model.tokenizer(chunks, verbose=False)["input_ids"]]

print(f"CS3244 alone: about {chunk_lengths[modules.index(by_code['CS3244'])]} tokens")
print(f"All {len(modules)} modules: about {sum(chunk_lengths):,} tokens, in every single prompt")
print(f"5 relevant modules: about {sum(chunk_lengths[:5]):,} tokens")

try:
    gemini_limit = rag_utils.gemini_client().models.get(model=rag_utils.GEMINI_MODEL).input_token_limit
    print(f"{rag_utils.GEMINI_MODEL} reads at most {gemini_limit:,} tokens per prompt")
except Exception:
    pass

All of NUSMods would fill most of the budget, if it fits at all, and every single question would pay for it again, when an answer needs only a handful of modules. **So retrieve only the most relevant modules, and let the LLM answer from those.** A RAG system does that in seven steps:

| Step | What happens | Where |
|---|---|---|
| 1. Encode, 2. Index | the embedding model turns every module into a vector, and we store the vectors | Section 2 |
| 3. Encode, 4. Similarity search | the query becomes a vector too, and we find the module vectors closest to it | Section 2 |
| 5. Similar documents | the top few modules come back as the retrieved context | Section 2 |
| 6. Prompt | the question and the retrieved modules go to the LLM together | Section 3 |
| 7. Final response | the LLM answers from them, citing its sources | Section 3 |

## Recap

So far: text → tokens → word counts (Bag of Words, TF-IDF, BM25) → find the best-matching modules → give them to the LLM (RAG).

But keyword methods only match the exact same words. Search for "artificial intelligence": the CS3244 description talks about "machine learning and neural networks", so the two share 0 words, and BM25 scores CS3244 at 0, even though it is clearly an AI module.

In [ ]:
cs3244 = modules.index(by_code["CS3244"])

print('BM25 score of CS3244 for "artificial intelligence":', word_scores("artificial intelligence", "bm25")[cs3244])
print('Ctrl+F matches for "how computers understand language":', len(keyword_search("how computers understand language")))

Word counts still do not capture meaning, which is why we need **embeddings**. That is where Section 2 starts.

### If you're stuck

Run the cell below. It reloads everything Section 2 needs.

In [ ]:
# Reloads the data and the embedding model if anything above failed

import matplotlib.pyplot as plt
import numpy as np

import rag_utils
from rag_utils import ask_gemini, build_chunk, keyword_search, search

modules, embeddings, model = rag_utils.load()
by_code = {module["moduleCode"]: module for module in modules}
print(len(modules), "modules, embeddings", embeddings.shape)

# Section 2 - Retrieval Engine

Section 1 ended with the RAG pipeline. Step 2, **retrieve**, has to find the few modules that answer a question, so that only those go into the prompt. Ctrl+F cannot do that job: "how computers understand language" found nothing, because Ctrl+F matches letters, and meaning is not in the letters.

This section builds a retriever that matches meaning, in three steps:

1. **Embedding:** turn text into numbers that capture meaning.
2. **Chunking:** decide how much text goes into each set of numbers.
3. **Retrieval:** find the modules closest to a question.

Cells marked **YOUR TURN** are for you to edit. Everything else runs as it is.

In [ ]:
# Two small helpers used throughout this section.
row_of = {module["moduleCode"]: i for i, module in enumerate(modules)}


def show(results):
    """Print (module, score) pairs, one per line."""
    for module, score in results:
        print(f"  {score:.2f}  {module['moduleCode']:<9} {module['title']}")

## 2.1 Embedding

### First try: words as vectors

Ctrl+F gives each module a yes or a no, with no ranking. The classic fix is **TF-IDF**: give every word in the vocabulary its own slot in a vector, and fill each slot with a count.

- **TF (term frequency):** how often the word appears in this module.
- **IDF (inverse document frequency):** how rare the word is across all modules. "the" appears everywhere, so it counts for almost nothing. "language" is much rarer, so it counts a lot.

Compare the query's vector with each module's vector and every module gets a score, so the results come back ranked. **BM25** is a tuned version of TF-IDF, and it is what most search engines still use.

But TF-IDF still matches words. "How do I learn to code?" and "Where can I pick up programming?" share no important word, so TF-IDF scores them 0. We need numbers that capture meaning.

### Embeddings: meaning as vectors

An **embedding model** reads a piece of text and returns a list of numbers, called a **vector**, that describes what the text is about. The numbers are not word counts: texts with similar meanings get similar vectors, even when they share no words.

We use `all-MiniLM-L6-v2`:

- **Small:** about 90 MB, and it runs on a laptop CPU.
- **Local:** no API key and no cost, because nothing is sent anywhere.
- **384 dimensions:** every text becomes 384 numbers, whether it is one word or a paragraph.

In [ ]:
vector = model.encode("Machine learning is fun")

print("Shape:", vector.shape)
print("First 8 numbers:", vector[:8].round(3))

The individual numbers mean nothing on their own. What matters is how close two vectors are, and that comes from how the model was built.

### How it works

1. **Split** the text into tokens: whole words, or pieces of words.
2. **Look up** a starting vector for each token.
3. **Mix:** the model's layers let each token read the others, so "bank" ends up with different numbers in "river bank" and in "bank account".
4. **Average** all the token vectors into one vector for the whole text.

Averaging alone does not create meaning. The model was trained on millions of pairs of texts, pulling similar pairs together and pushing unrelated ones apart. That training is what makes "close" mean "similar".

Two numbers not to mix up:

- **384 out:** every text becomes exactly 384 numbers, whether it is 5 words or 200.
- **256 in:** MiniLM reads at most 256 tokens. Anything after that is silently dropped, which matters in 2.2.

Here is step 1. A word the tokenizer does not know as a whole is split into pieces, and every piece after the first starts with `##`.

In [ ]:
print(model.tokenizer.tokenize("NUSMods lists every module"))

### Cosine similarity

Picture each vector as an arrow. To compare two texts, we look at the angle between their arrows:

- **Same direction:** about 1, very similar.
- **Right angle:** about 0, unrelated.

This score is called **cosine similarity**. It is the **dot product** (multiply the two vectors position by position, then add everything up) divided by both lengths, so only the direction counts:

$$\cos(\theta) = \frac{A \cdot B}{\|A\| \, \|B\|}$$

In [ ]:
def cosine(a, b):
    """Cosine similarity: the dot product divided by both lengths."""
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))


sentences = [
    "How do I learn to code?",
    "Where can I pick up programming?",
    "What should I eat for lunch?",
]
vectors = model.encode(sentences)

print("Sentence 1 vs 2:", round(cosine(vectors[0], vectors[1]), 2))
print("Sentence 1 vs 3:", round(cosine(vectors[0], vectors[2]), 2))

Sentences 1 and 2 share no important word, so TF-IDF scores them 0. Their vectors still score far higher than 1 and 3.

### Real modules

Every module already has a vector. They were computed in advance and stored in `embeddings`, where row `i` belongs to `modules[i]`. They were also scaled to length 1, so the plot below can skip the division in the formula. 2.3 explains why that helps.

Here are 10 modules from four areas, each compared with all the others.

In [ ]:
def plot_similarity(labels, vectors, title):
    """Heatmap of the similarity between every pair of vectors."""
    similarity = vectors @ vectors.T
    count = len(labels)

    fig, ax = plt.subplots(figsize=(9, 6))
    image = ax.imshow(similarity, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(count), range(1, count + 1))
    ax.set_yticks(range(count), [f"{label}  {i + 1}" for i, label in enumerate(labels)])
    for i in range(count):
        for j in range(count):
            colour = "white" if similarity[i, j] > 0.6 else "black"
            ax.text(j, i, f"{similarity[i, j]:.2f}", ha="center", va="center", fontsize=8, color=colour)
    fig.colorbar(image, label="Similarity")
    ax.set_title(title)
    plt.show()


codes = [
    "CS2109S", "CS3244", "CS4248",     # AI
    "EC1101E", "FIN2704", "FIN3702A",  # economics and finance
    "LSM1301", "LSM2105",              # biology
    "EN1101E", "EN2210",               # literature
]
rows = [row_of[code] for code in codes]
labels = [f"{code} {modules[row]['title']}" for code, row in zip(codes, rows)]

plot_similarity(labels, embeddings[rows], "How similar are these modules?")

**How to read it:** each cell compares two modules, and a darker cell means more similar. The diagonal is every module against itself, so it is always 1.00.

The darker squares along the diagonal are the four areas: AI, economics and finance, biology, and literature. Nobody told the model which faculty a module belongs to. It grouped them from the text alone.

Unrelated modules do not score 0. They mostly sit around 0.1 to 0.3, so read the scores as a ranking: related pairs score higher than unrelated ones.

**Think about it:** find CS4248, the large language models module. How does it score against the two literature modules, compared with FIN2704? What might the model be picking up?

**YOUR TURN:** edit the sentences below and re-run the cell. Which pairs surprise you? Some things to try:

- The same word with two meanings: "I sat by the river bank" and "I opened a bank account".
- Opposites: "I love durian" and "I hate durian".
- The same sentence in two languages.

In [ ]:
# YOUR TURN: edit these sentences, then re-run
my_sentences = [
    "I love machine learning",
    "Neural networks are fascinating",
    "The stock market crashed today",
    "Investors are worried about inflation",
    "My cat sleeps all day",
    "Dogs are loyal pets",
]

plot_similarity(my_sentences, model.encode(my_sentences, normalize_embeddings=True), "My sentences")

## 2.2 Chunking

We can now turn any text into a vector. But how much text should go into each vector: a whole handbook, one page, or one sentence?

A **chunk** is one piece of text that gets its own vector. Real sources such as a handbook, a PDF or a website are too long to embed in one go, so we split them. There are two reasons:

- **Token budget:** MiniLM reads at most 256 tokens. Anything after that is silently cut off.
- **Precision:** one vector has to summarise the whole chunk. A vector for a 50-page handbook is about everything, so it is not close to any particular question.

Too small hurts too. A chunk that holds only a workload line has lost the module it belongs to. Aim for one self-contained idea per chunk.

Common ways to split:

- **Fixed size:** cut every N tokens.
- **Fixed size with overlap:** neighbouring chunks share some tokens, so a sentence cut at one boundary still appears whole in the next chunk.
- **By structure:** cut at natural boundaries such as sections or paragraphs.

NUSMods is a lucky case. The data is already split by structure, and each module is short and self-contained, so **one module = one chunk**. The decision left is which fields of a module go into the chunk text.

In [ ]:
module = modules[row_of["CS2040S"]]

print("The raw module:")
for field, value in module.items():
    print(f"  {field}: {value}")

print()
print("The chunk that gets embedded:")
print(build_chunk(module))

`build_chunk` (in `rag_utils.py`) joins four fields with line breaks: code, title, description and workload.

Look at the workload. NUSMods stores it as `[3, 0, 1, 3, 3]`, which tells an embedding model nothing, so we write it out in words first. Preparing text so that the model can use it is part of chunking.

How close are we to the token budget?

In [ ]:
chunks = [build_chunk(module) for module in modules]
lengths = [len(ids) for ids in model.tokenizer(chunks, verbose=False)["input_ids"]]

print("CS2040S chunk:", lengths[row_of["CS2040S"]], "tokens. The limit is", model.max_seq_length)
print("Typical chunk:", int(np.median(lengths)), "tokens")

too_long = sum(length > model.max_seq_length for length in lengths)
print(f"{too_long} of {len(modules)} chunks are over the limit and lose their ending")

Only a small share of chunks get cut. When it happens, the end of the description and the workload are lost. That is why the code and title go first.

### What we left out

NUSMods has 21,160 modules on record for AY 2026-2027. We kept 6,552:

- **14,382 dropped** because they are not offered this year.
- **226 dropped** because the description is a placeholder such as "Not Available". With nothing to describe the module, its vector would come from the code and title alone.

### Which fields go in?

Adding a field is not free. To see why, we embed the same modules twice: once with the default chunk, and once with the prerequisites added.

Re-embedding all 6,552 modules takes minutes on a CPU, so we use the Computing modules only.

In [ ]:
computing = [module for module in modules if module["faculty"] == "Computing"]

fields_default = ["moduleCode", "title", "description", "workload"]
fields_prereq = fields_default + ["prerequisite"]

vectors_default = model.encode([build_chunk(m, fields_default) for m in computing], normalize_embeddings=True)
vectors_prereq = model.encode([build_chunk(m, fields_prereq) for m in computing], normalize_embeddings=True)

print(len(computing), "Computing modules embedded twice")
print()
print("A chunk with prerequisites:")
print(build_chunk(module, fields_prereq))

In [ ]:
def compare_chunks(query, needs=None):
    """Search the Computing modules with both versions of the chunk.

    `needs` is a module code. Results that list it as a prerequisite get a tick.
    """
    for name, vectors in [("Default chunk", vectors_default), ("With prerequisites", vectors_prereq)]:
        print(name)
        for result, score in search(query, embeddings=vectors, modules=computing):
            tick = "✓" if needs and needs in result.get("prerequisite", "") else " "
            print(f"  {score:.2f} {tick} {result['moduleCode']:<9} {result['title']}")


compare_chunks("I have completed CS2040S, what next?", needs="CS2040S")

A tick marks a module that really does list CS2040S as a prerequisite. With the prerequisites in the chunk, more of the top 5 are correct, because the text the question is about is now there to be matched.

Now a question about a topic.

In [ ]:
compare_chunks("I want to learn machine learning")

Every score dropped, and CS3244 Machine Learning, the module with exactly that title, fell out of the top 5. Its prerequisite text is long, and those words now share the vector with the topic.

That is the tradeoff: a field you add helps the questions about that field and dilutes everything else. Longer chunks also push more modules past the 256-token limit. Most questions are about topics, so the default chunk leaves prerequisites out.

**YOUR TURN:** choose your own fields below, re-embed the Computing modules, and compare with the default chunk. Does a shorter chunk do better or worse on your question?

In [ ]:
# YOUR TURN: choose the fields and the question, then re-run
# Fields you can use: moduleCode, title, description, workload, prerequisite, faculty, department
my_fields = ["title", "description"]
my_question = "I want to learn machine learning"

my_vectors = model.encode([build_chunk(m, my_fields) for m in computing], normalize_embeddings=True)

print("Default chunk")
show(search(my_question, embeddings=vectors_default, modules=computing))
print("My chunk:", my_fields)
show(search(my_question, embeddings=my_vectors, modules=computing))

## 2.3 Retrieval

We have chunks and their vectors. Now: which chunks best answer a question?

1. Embed every chunk once, ahead of time. That is `embeddings`.
2. Embed the query with the same model.
3. Compare the query with every module vector.
4. Sort by score and take the top `k`.

Step 1 is why search is fast: when a question comes in, only the query gets embedded.

In [ ]:
def my_search(query, k=5):
    query_vector = model.encode(query, normalize_embeddings=True)  # 2. embed the query
    scores = embeddings @ query_vector                             # 3. compare with every module
    top = np.argsort(-scores)[:k]                                  # 4. sort, take the top k
    return [(modules[i], float(scores[i])) for i in top]

That is the whole retrieval engine, and it is what `search` in `rag_utils.py` does.

**Why `embeddings @ query_vector` is cosine similarity:** with `normalize_embeddings=True`, every vector has length 1. The bottom of the cosine formula becomes 1, so cosine similarity is just the dot product. The line `embeddings @ query_vector` is one matrix multiplication that computes all 6,552 dot products at once. This only works because the module vectors were normalized too.

### Ctrl+F, again

Back to the query that returned nothing in Section 1.

In [ ]:
query = "how computers understand language"

print("Keyword search:", len(keyword_search(query)), "matches")
print("My search:")
show(my_search(query))

Keyword search finds nothing, because no description contains that exact phrase. Your search returns modules about language, and the top hit is a course on how humans and machines handle language.

**Think about it:** most of the results are linguistics modules, not computing ones. Which words in the query pulled them in?

### Do we need a vector database?

In [ ]:
import time

query_vector = model.encode("how computers understand language", normalize_embeddings=True)

start = time.perf_counter()
scores = embeddings @ query_vector
top = np.argsort(-scores)[:5]
elapsed_ms = (time.perf_counter() - start) * 1000

print(f"Compared the query with {len(modules)} modules in {elapsed_ms:.1f} ms")

No. A vector database earns its place at millions of chunks, where comparing against everything becomes too slow. For a few thousand chunks, plain numpy is enough.

### Limits

Semantic search won on the opening query. Now a query where the exact word matters, with keyword search and semantic search side by side.

In [ ]:
def compare(query, k=5):
    """Show keyword search and semantic search for the same query."""
    matches = keyword_search(query)
    print(f"Keyword search: {len(matches)} matches")
    for match in matches[:k]:
        print(f"        {match['moduleCode']:<9} {match['title']}")
    print("Semantic search:")
    show(search(query, k))


compare("ChatGPT")

Keyword search wins this one. Four descriptions mention ChatGPT by name, and Ctrl+F finds all of them. Semantic search finds one, then fills the other slots with communication modules.

Why? MiniLM was trained before ChatGPT existed, so it has never seen the word. This is how its tokenizer reads it:

In [ ]:
print(model.tokenizer.tokenize("ChatGPT"))

The model sees "chat" plus two fragments, the same splitting you saw in 2.1, so it goes looking for modules about communication. The embedding model has a knowledge cutoff too, just like the LLM in Section 1.

| | Keyword search | Semantic search |
|---|---|---|
| Matches | exact words | similar meaning |
| Good at | names, rare terms, new words | paraphrases and vague questions |
| Fails when | you use different words from the text | the exact word matters, or the model has never seen it |
| Returns | every match, in no particular order | always `k` results, ranked by score |

The last row matters: semantic search **always** returns `k` modules, even when nothing relevant exists.

### Hybrid search

Each method fails exactly where the other is strong. Remember BM25 from 2.1: it ranks results like semantic search, but it scores exact words, so it handles codes, names and new words such as ChatGPT. Real systems run both and merge the two rankings, so a module comes out on top if it matches the words or the meaning. This is called **hybrid search**.

Our engine uses semantic search only, so expect it to miss exact words.

**YOUR TURN:** ask your own questions. Some things to try:

- A topic in your own words, the way you would ask a senior.
- A module code, such as `CS2040S`.
- A greeting, such as `hi`.
- A filter, such as `4 MC, no exam, semester 2`.
- A larger `k`. Do the extra results still look relevant?

**Keep your funniest failure.** We will collect them in Section 3.

In [ ]:
# YOUR TURN: ask your own question, and change k
my_query = "I want to learn how to invest"
k = 5

compare(my_query, k)

## Recap

- **Words as vectors:** TF-IDF and BM25 rank modules by shared words, so a paraphrase with different words scores 0.
- **Embedding:** a model turns text into a vector, similar meanings land close together, and cosine similarity measures how close.
- **Chunking:** one module is one chunk, and the fields you put in the chunk decide what it can be found by.
- **Retrieval:** embed the query, take a dot product with every module, sort, and keep the top `k`.
- **Hybrid search:** real systems combine BM25 for exact words with embeddings for meaning.

Section 3 takes the modules returned by `search(query, k=5)` and hands them to Gemini.

### If you're stuck

Run the cell below. It reloads everything Section 3 needs.

In [ ]:
# Reloads the data, the embedding model, Gemini and the helpers from Sections 1 and 2 if anything above failed
import re
import textwrap

from IPython.display import Markdown, display

import rag_utils
from rag_utils import ask_gemini, search

modules, embeddings, model = rag_utils.load()
by_code = {module["moduleCode"]: module for module in modules}
rag_utils.connect_gemini()

# From Section 1.5
MODULE_CODE = re.compile(r"\b[A-Z]{2,4}\d{4}[A-Z]{0,2}\b")  # e.g. CS4248, CS2103T

def verify(answer):
    """Look up every module code in the answer in the real NUSMods data."""
    codes = list(dict.fromkeys(MODULE_CODE.findall(answer)))  # unique, in order
    for code in codes:
        real = by_code.get(code)
        if real:
            print(f"  ✅ {code:<9} real title: {real['title']}")
        else:
            print(f"  ❌ {code:<9} not offered in AY {rag_utils.AY}, or does not exist")
        said = next(line for line in answer.splitlines() if code in line)  # where Gemini mentioned it
        print(f"{'':15}Gemini wrote: {textwrap.shorten(re.sub(r'[*_#`|>]', '', said), 90)}")
    print()
    print(f"{sum(code in by_code for code in codes)} of {len(codes)} module codes are real.")

# From Section 2
def show(results):
    """Print (module, score) pairs, one per line."""
    for module, score in results:
        print(f"  {score:.2f}  {module['moduleCode']:<9} {module['title']}")

print(len(modules), "modules, embeddings", embeddings.shape)

# Section 3 - Generation + Evaluation

Section 2 built the retriever. This section finishes the RAG pipeline from Section 1.6, then checks how well the retriever works:

1. **Generation:** paste the retrieved modules into the prompt (step 3, *augment*) and let Gemini answer from them (step 4, *generate*).
2. **Evaluation:** measure how often the retriever finds the right module, because the answer can only be as good as what was retrieved.

## 3.1 Generation

Two small functions do the augment step:

- `make_context` turns the `(module, score)` pairs from `search` into plain text, one numbered block per module.
- `build_prompt` wraps that text in instructions: answer **only** from the context, say so when the context is not enough, and cite module codes in square brackets.

In [ ]:
def make_context(results):
    """Format retrieved (module, score) pairs as numbered text blocks."""
    blocks = []
    for rank, (module, score) in enumerate(results, start=1):
        lines = [
            f"[{rank}] {module['moduleCode']} - {module['title']}",
            f"Description: {module['description']}",
        ]
        workload = rag_utils.format_workload(module.get("workload"))
        if workload:
            lines.append(f"Workload: {workload}")
        if module.get("semesters"):
            lines.append(f"Semesters offered in AY {rag_utils.AY}: {', '.join(map(str, module['semesters']))}")
        if module.get("hasExam") is not None:
            lines.append(f"Has final exam: {'yes' if module['hasExam'] else 'no'}")
        lines.append(f"Retrieval similarity: {score:.3f}")
        blocks.append("\n".join(lines))
    return "\n\n".join(blocks) if blocks else "No modules were retrieved."


def build_prompt(question, context):
    """Wrap the question and the retrieved context in grounding instructions."""
    return f"""You are a helpful assistant answering questions about NUS modules.
Answer the user's question using only the supplied NUSMods context.

Rules:
- If the context does not contain enough evidence, say that the information is not available in the supplied context.
- Do not invent module codes, titles, prerequisites, workload hours, semesters, or exam details.
- When making a claim about a module, cite its module code in square brackets, e.g. [CS1231S].
- Keep the answer concise and distinguish explicit facts from uncertainty.
- The retrieval similarity score is not evidence that a claim is true.

Retrieved context:
{context}

User question: {question}

Answer:"""


def rag_answer(question, k=5, results=None, show_prompt=False):
    """Retrieve, augment, generate. Pass `results` to supply your own context."""
    if results is None:
        results = search(question, k=k)
    prompt = build_prompt(question, make_context(results))
    if show_prompt:
        print(prompt)
        print("-" * 60)
    answer = ask_gemini(prompt)
    display(Markdown(answer))
    return answer

### The same question, now with RAG

In Section 1.5, Gemini answered this question from memory, and `verify` caught the module codes it got wrong. Here is the exact prompt RAG sends for the same question. Gemini sees only this text.

In [ ]:
question = "Which NUS modules cover attention mechanisms? List the module codes and titles."

print(build_prompt(question, make_context(search(question, k=5))))

In [ ]:
answer = rag_answer(question)
print("-" * 60)
verify(answer)

Every module code Gemini can cite now comes from the context, so `verify` should find each one in the real data. Compare this with the 1.5 result.

That fixes invented codes, but not missing ones. Look back at the retrieved modules in the prompt: are they about attention in neural networks, or attention in psychology? If the right module was not retrieved, Gemini cannot mention it. The answer is only as good as the context.

### Wrong context, confident answer

To see how much the answer depends on retrieval, keep the question and swap in modules retrieved for an unrelated topic.

In [ ]:
wrong_results = search("modules about economics and financial markets", k=5)
show(wrong_results)
print("-" * 60)

wrong_answer = rag_answer(question, results=wrong_results)

**How to read it:** the prompt tells Gemini to say when the context is not enough. Check whether it did, or whether it stretched an economics module to fit a question about attention mechanisms. Either way, generation cannot recover from bad retrieval, which is why 3.2 measures the retriever on its own.

**YOUR TURN:** ask your own question. Set `show_prompt=True` to see exactly what Gemini received, and change `k` to give it more or fewer modules.

In [ ]:
# YOUR TURN: ask your own question, then re-run
my_question = "Which NUS modules teach reinforcement learning? List the module codes and titles."

my_answer = rag_answer(my_question, k=5, show_prompt=False)
print("-" * 60)
verify(my_answer)

## 3.2 Evaluation

Now we evaluate how effectively our RAG system retrieves the correct NUS module for a user's question.

For each query, we specify the expected module code and compare it with the modules returned by our retrieval system.

We use two metrics:

- **Hit@5:** Measures the proportion of queries for which the correct module appears in the top five retrieved results.
- **Mean Reciprocal Rank (MRR):** Measures how highly the correct module is ranked, rewarding results that appear closer to the top.

These metrics help us assess retrieval performance before investigating potential improvements.

In [ ]:
import pandas as pd

workshop_dir = rag_utils.DATA_DIR.parent  # the week-11-nlp folder

In [ ]:
eval_path = workshop_dir / "data" / "eval_set_v2.csv"

eval_df = pd.read_csv(eval_path)

print(f"Total evaluation queries: {len(eval_df)}")
print("\nQueries by category:")
display(eval_df["category"].value_counts().to_frame("count"))

display(eval_df.head())

In [ ]:
def evaluate_multilabel_retrieval(eval_df, k=5):
    details = []

    for _, row in eval_df.iterrows():
        query = str(row["query"])
        relevant_codes = {
            code.strip().upper()
            for code in str(row["relevant"]).split("|")
            if code.strip()
        }

        results = rag_utils.search(query, k=k)

        ranked_codes = [
            str(module.get("moduleCode", "")).upper()
            for module, score in results
        ]

        # Rank of the first retrieved relevant module
        relevant_ranks = [
            rank for rank, code in enumerate(ranked_codes, start=1)
            if code in relevant_codes
        ]

        first_relevant_rank = (
            min(relevant_ranks) if relevant_ranks else 0
        )

        details.append({
            "query": query,
            "category": row["category"],
            "relevant_codes": " | ".join(sorted(relevant_codes)),
            "top_k_module_codes": " | ".join(ranked_codes),
            "rank_of_first_relevant": first_relevant_rank,
            f"hit_at_{k}": int(bool(relevant_ranks)),
            "reciprocal_rank": (
                1 / first_relevant_rank
                if first_relevant_rank else 0
            )
        })

    details_df = pd.DataFrame(details)

    summary = {
        "n_queries": len(details_df),
        f"hit_at_{k}": details_df[f"hit_at_{k}"].mean(),
        "mrr": details_df["reciprocal_rank"].mean()
    }

    return summary, details_df


In [ ]:
summary, details_df = evaluate_multilabel_retrieval(
    eval_df,
    k=5
)

print(f"Queries evaluated: {summary['n_queries']}")
print(f"Hit@5: {summary['hit_at_5']:.3f}")
print(f"MRR: {summary['mrr']:.3f}")

results_dir = workshop_dir / "evaluation"
results_dir.mkdir(parents=True, exist_ok=True)

output_path = results_dir / "eval_set_v2_results.csv"
details_df.to_csv(output_path, index=False)

print(f"Results saved to: {output_path}")

display(details_df)

**YOUR TURN:** pick a query from the table above that missed (`hit_at_5` is 0), or ask your own question, and look at what the retriever returned instead.

In [ ]:
# YOUR TURN: change the question, then re-run
query = "Which NUS modules cover attention mechanisms?"

results = rag_utils.search(query, k=5)

print(f"Query: {query}\n")
print("Top 5 retrieved modules:\n")

for rank, (module, score) in enumerate(results, start=1):
    print(f"{rank}. {module.get('moduleCode', 'N/A')} - "
          f"{module.get('title', 'N/A')}")
    print(f"   Similarity score: {score:.4f}\n")

## Recap

- **Generation:** the retrieved modules go into the prompt as context, and the instructions keep Gemini to that context and make it cite module codes.
- **Evaluation:** Hit@5 asks whether a relevant module made the top 5; MRR rewards finding it near the top.
- **The weak link is retrieval:** with the wrong context, even a good LLM gives a poor answer.